### Run Dependencies

In [ ]:
%run Legal_-_Libraries_And_Path
%run Legal_-_Logs_Warehouse


### Spark Config

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


### List RAW Layer Files

In [ ]:
# ── List all CSV files in RAW_LAYER ───────────────────────────────────────────
File_Path = mssparkutils.fs.ls(Landing_Folder_Path)
CSV_Files = [f.path for f in File_Path if f.path.endswith(".csv")]

print(f"Found {len(CSV_Files)} CSV file(s) in RAW_LAYER:")
for _f in CSV_Files:
    print(f"  {_f}")


### Load & Cache Schema File  |  Build Lookup Dicts

In [ ]:
# ── Schema CSV location ────────────────────────────────────────────────────────
Schema_Data_File_Location = f"{Lakehouse_Folder_Storage_Path}/Legal_Schema.csv"

# ── Read schema mapping file and CACHE it (avoids repeated Spark jobs) ────────
Schema_File = (
    spark.read.csv(Schema_Data_File_Location, header=True, inferSchema=True, sep=",")
    .cache()
)

print(f"Schema_File loaded and cached | rows={Schema_File.count()}")

# ── Build column-type lookup dict  (file_name, dtype) → [col1, col2, ...] ─────
#    This replaces the 5 × N individual .filter().collect() calls per table.
from collections import defaultdict

_schema_rows      = Schema_File.collect()
schema_lookup     = defaultdict(list)          # (file_name, dtype)  → [col_names]
pk_lookup         = {}                         # file_name → primary_key col
uk_lookup         = {}                         # file_name → [unique_key_cols]

for _row in _schema_rows:
    _fn    = _row["File_Name"]
    _dtype = _row["Sink_Dtypes"]
    _col   = _row["Sink_Columns"]
    schema_lookup[(_fn, _dtype)].append(_col)

    if _row["Primary_Key"] and _fn not in pk_lookup:
        pk_lookup[_fn] = _row["Primary_Key"]

    if _row["Unique_Key_Combination"] and _fn not in uk_lookup:
        uk_lookup[_fn] = [k.strip() for k in _row["Unique_Key_Combination"].split(",")]

print(f"schema_lookup built | {len(schema_lookup)} entries")
print(f"pk_lookup    built  | {len(pk_lookup)} tables with a PK defined")
print(f"uk_lookup    built  | {len(uk_lookup)} tables with UK defined")


### Schema Type Mappings  |  `build_spark_schema()` Helper

In [ ]:
# ── Dtype string → Spark type mapping ────────────────────────────────────────
#    Bronze reads everything as StringType (raw landing).
#    Silver uses the real types from Schema_File.
Schema_Mapping_Bronze = {
    "IntegerType()"  : StringType(),
    "StringType()"   : StringType(),
    "DateType()"     : StringType(),
    "LongType()"     : StringType(),
    "FloatType()"    : StringType(),
    "TimestampType()": StringType(),
}

Schema_Mapping_Silver = {
    "IntegerType()"  : IntegerType(),
    "StringType()"   : StringType(),
    "DateType()"     : DateType(),
    "LongType()"     : LongType(),
    "FloatType()"    : FloatType(),
    "TimestampType()": TimestampType(),
}


def build_spark_schema(file_name: str, mapping: dict) -> StructType:
    """
    Builds a Spark StructType from Schema_File for the given file_name.
    Uses the pre-built schema_lookup dict — zero extra Spark jobs.

    Parameters
    ----------
    file_name : e.g. "raw_lawyer_profile.csv"
    mapping   : Schema_Mapping_Bronze or Schema_Mapping_Silver
    """
    rows = [r for r in _schema_rows if r["File_Name"] == file_name]
    if not rows:
        raise ValueError(f"No schema entries found for {file_name}")
    return StructType([
        StructField(r["Sink_Columns"], mapping[r["Sink_Dtypes"]], True)
        for r in rows
    ])


print("build_spark_schema() registered.")


### Audit Log — RAW Validation

In [ ]:
import uuid as _uuid_mod

log_audit(
    audit_id          = str(_uuid_mod.uuid4()),
    source_type       = "RAW_VALIDATION",
    destination       = f"{Container_Name}/{Landing_Folder_Name}",
    notebook_name     = "Legal - File_Formats_And_Processing",
    layer_name        = "RAW",
    table_name        = "ALL_RAW_FILES",
    records_processed = len(CSV_Files),
    status            = "SUCCESS",
    error_message     = "",
)
